In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
import warnings
warnings.filterwarnings('ignore')

# Models for denoising task
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

num_terms = 78
class DenoisingModelSelector:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.models = {}
        self.best_model = None
        self.best_score = float('inf')
        self.results = {}
        
    def prepare_data(self, X, y, test_size=0.2):
        """Prepare and scale the data"""
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        return X_train_scaled, X_test_scaled, y_train, y_test
    
    def get_model_configs(self):
        """Define models and their hyperparameter grids"""
        model_configs = {
            'RandomForest': {
                'model': RandomForestRegressor(random_state=self.random_state),
                'params': {
                    'n_estimators': [50, 100, 200],
                    'max_depth': [None, 10, 20],
                    'min_samples_split': [2, 5]
                }
            },
            'XGBoost': {
                'model': XGBRegressor(random_state=self.random_state),
                'params': {
                    'n_estimators': [50, 100],
                    'max_depth': [3, 6],
                    'learning_rate': [0.05, 0.1]
                }
            },
            'Ridge': {
                'model': Ridge(random_state=self.random_state),
                'params': {
                    'alpha': [0.1, 1.0, 10.0]
                }
            },
            'Lasso': {
                'model': Lasso(random_state=self.random_state),
                'params': {
                    'alpha': [0.1, 1.0, 10.0]
                }
            },
            'ElasticNet': {
                'model': ElasticNet(random_state=self.random_state),
                'params': {
                    'alpha': [0.1, 1.0],
                    'l1_ratio': [0.2, 0.5, 0.8]
                }
            },
            'KNN': {
                'model': KNeighborsRegressor(),
                'params': {
                    'n_neighbors': [3, 5, 7],
                    'weights': ['uniform', 'distance']
                }
            },
            'MLP': {
                'model': MLPRegressor(random_state=self.random_state, max_iter=1000),
                'params': {
                    'hidden_layer_sizes': [(50,), (100,)],
                    'alpha': [0.0001, 0.001],
                    'learning_rate_init': [0.001, 0.01]
                }
            }
        }
        return model_configs
    
    def evaluate_model(self, model, X_test, y_test):
        """Comprehensive model evaluation"""
        y_pred = model.predict(X_test)
        
        mse = np.mean(((y_test - y_pred).flatten())**2)
        rsme = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
               
        
        metrics = {
            'MSE': mse,
            'RMSE': rsme,
            'MAE': mae,
        }
        
        return metrics
    
    def select_best_model(self, X, y, cv=5, scoring='neg_mean_squared_error'):
        """Main method to select the best model"""
        print("Starting model selection for denoising task...")
        
        # Prepare data
        X_train, X_test, y_train, y_test = self.prepare_data(X, y)
        
        model_configs = self.get_model_configs()
        self.results = {}
        
        for name, config in model_configs.items():
            print(f"\n--- Training {name} ---")
            
            try:
                # Grid search with cross-validation
                grid_search = GridSearchCV(
                    config['model'], config['params'], 
                    cv=cv, scoring=scoring, n_jobs=-1, verbose=0
                )
                
                grid_search.fit(X_train, y_train)
                
                # Store best model
                self.models[name] = grid_search.best_estimator_
                
                # Evaluate on test set
                test_metrics = self.evaluate_model(grid_search.best_estimator_, X_test, y_test)
                
                self.results[name] = {
                    'best_params': grid_search.best_params_,
                    'test_metrics': test_metrics,
                    'cv_score': grid_search.best_score_
                }
                
                print(f"Best params: {grid_search.best_params_}")
                print(f"Test MSE: {test_metrics['MSE']:.4f}, MAE: {test_metrics['MAE']:.4f}")
                
                # Update best model
                if test_metrics['MSE'] < self.best_score:
                    self.best_score = test_metrics['MSE']
                    self.best_model = grid_search.best_estimator_
                    self.best_model_name = name
                    
            except Exception as e:
                print(f"Error with {name}: {str(e)}")
                continue
        
        return self.best_model
    
    def print_results(self):
        """Print comprehensive results"""
        print("\n" + "="*60)
        print("MODEL SELECTION RESULTS")
        print("="*60)
        
        # Sort models by test MSE
        sorted_results = sorted(
            self.results.items(), 
            key=lambda x: x[1]['test_metrics']['MSE']
        )
        
        for name, result in sorted_results:
            metrics = result['test_metrics']
            print(f"\n{name}:")
            print(f"  MSE: {metrics['MSE']:.4f}")
            print(f"  RMSE: {metrics['RMSE']:.4f}")
            print(f"  MAE: {metrics['MAE']:.4f}")
            print(f"  Best params: {result['best_params']}")
        
        print(f"\n🏆 BEST MODEL: {self.best_model_name}")
        print(f"Best Test MAE: {self.best_score:.4f}")

# Usage example
def main(X, y):
    # Initialize selector
    selector = DenoisingModelSelector(random_state=42)
    
    # Run model selection
    best_model = selector.select_best_model(X, y, cv=5)
    
    # Print results
    selector.print_results()
    
    return selector

In [4]:
df_near = pd.read_csv('./data/datasets/near_depolarization01.csv')
df = pd.read_csv('./data/datasets/depolarization01.csv')

X_near = df_near[[f'noisy_{n}' for n in range(num_terms)]].values
y_near = df_near[[f'target_{n}' for n in range(num_terms)]].values

X = df[[f'noisy_{n}' for n in range(num_terms)]].values
y = df[[f'target_{n}' for n in range(num_terms)]].values

main(X_near, y_near)
main(X, y)

noisy_near = np.mean((((X_near - y_near).flatten())**2))
noisy = np.mean((((X - y).flatten())**2))

print(f"NOISY RMSE NEAR-CLIFFORD: {np.sqrt(noisy_near)}")
print(f"NOISY RMSE CLIFFORD: {np.sqrt(noisy)}")

Starting model selection for denoising task...

--- Training RandomForest ---
Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Test MSE: 0.0058, MAE: 0.0253

--- Training XGBoost ---
Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
Test MSE: 0.0006, MAE: 0.0057

--- Training Ridge ---
Best params: {'alpha': 10.0}
Test MSE: 0.0004, MAE: 0.0073

--- Training Lasso ---
Best params: {'alpha': 0.1}
Test MSE: 0.0053, MAE: 0.0209

--- Training ElasticNet ---
Best params: {'alpha': 0.1, 'l1_ratio': 0.2}
Test MSE: 0.0010, MAE: 0.0102

--- Training KNN ---
Best params: {'n_neighbors': 7, 'weights': 'distance'}
Test MSE: 0.0072, MAE: 0.0276

--- Training MLP ---
Best params: {'alpha': 0.001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.01}
Test MSE: 0.0052, MAE: 0.0424

MODEL SELECTION RESULTS

Ridge:
  MSE: 0.0004
  RMSE: 0.0188
  MAE: 0.0073
  Best params: {'alpha': 10.0}

XGBoost:
  MSE: 0.0006
  RMSE: 0.0240
  MAE: 0.0057
  Best para